# Headline batch recipe

Interactive end-to-end orchestrator for the experiment-tweaks headline batch.
**This notebook never fires `pixi`/`kubectl` itself** — every external command is
shown as a `bash` block (commented in the notes) for you to run in a terminal.
Between commands you come back here, re-run the inspection cells, and decide
what's next.

Three phases:

1. **Stage 1** — submit 5 reps × 9 models × 3 targets = 135 cells against
   `orchestration/matrix.csv`.
2. **Re-rep loop** — for each (model, target) combo that has fewer than
   `TARGET_N` cells with a `scorer.json`, generate `matrix-rerep.csv` and
   submit a new batch. Repeat until every combo has `TARGET_N` actually-ran
   cells. *This is also where olmo joins the panel* (5 reps × 3 targets).
3. **Stage 2** — once every combo has `TARGET_N` cells, identify which got
   ≥1 full-pass and re-fire them with `+10` reps each via `matrix-stage2.csv`.

After stage 2 completes, aggregate every batch tag together and render
[`analysis/headline.ipynb`](headline.ipynb) for the figures.

In [1]:
from pathlib import Path
import pandas as pd

REPO_ROOT = Path('..').resolve()

# === edit these as the batch progresses ===
IMAGE_SHA = 'a5b3eae'                                  # the SHA stage 1 was built against
STAGE1_TAG = 'headline-stage1-20260601'                # the initial 135-cell batch tag
TARGET_N = 5                                           # minimum scorer.json cells per combo before phase 2

# Full panel (olmo added 2026-06-01; joins via re-rep).
MODEL_PANEL = [
    # olmo (allenai/olmo-3.1-32b-instruct) dropped 2026-06-02 — every cell failed.
    'sonnet','kimi','qwen','deepseek','glm','minimax','gemma','mistral','nemotron',
]
TARGETS = ['josh','mesa','josh-mcp']

# Track every batch tag launched so the smell-check + stage-2 builder can
# aggregate across them. Append re-rep tags here as you go.
BATCH_TAGS = [STAGE1_TAG]

print(f'panel: {len(MODEL_PANEL)} models × {len(TARGETS)} targets = {len(MODEL_PANEL)*len(TARGETS)} combos')
print(f'target N per combo: {TARGET_N}')
print(f'image SHA: {IMAGE_SHA}')
print(f'batch tags: {BATCH_TAGS}')

panel: 9 models × 3 targets = 27 combos
target N per combo: 5
image SHA: a5b3eae
batch tags: ['headline-stage1-20260601']


## Phase 1 — submit stage 1

```bash
BATCH_TAG=headline-stage1-$(date -u +%Y%m%d)
pixi run apply -- \
    --batch-tag "$BATCH_TAG" \
    --image-agent  ghcr.io/schmidtdse/josh-llm-experiment/fortree-agent:a5b3eae \
    --image-scorer ghcr.io/schmidtdse/josh-llm-experiment/fortree-scorer:a5b3eae \
    --matrix orchestration/matrix.csv

# Watch the cells reach terminal state:
kubectl -n joshsim get jobs -l batch-tag=$BATCH_TAG -w

# Pull artefacts + aggregate when done:
pixi run pull $BATCH_TAG
pixi run aggregate runs/$BATCH_TAG
```

Once `aggregate` writes `analysis/aggregated.csv`, run the cells below to
inspect state and generate the re-rep matrix.

## Phase 2 — re-rep loop until every combo has `TARGET_N` cells

**What counts as a "ran" cell**: anything where the aggregator wrote a row
with `engagement_status != 'no_scorer'`. The `no_scorer` rows are synthetic —
cells that died at `FailJob` before the agent produced output. Those need to
re-rep. Any other engagement status (`partial_steps`, `full_steps_no_files`,
`full_steps_with_files`) means the agent at least started and counts toward
the `TARGET_N` budget, regardless of pass/fail.

Olmo joins here: the panel has 10 models but stage 1 only ran 9, so olmo
naturally shows up with 0 cells and gets 5 reps × 3 targets emitted by the
re-rep builder.

In [2]:
# Helpers — kept in one cell so the recipe is greppable.

def load_panel(batch_tags):
    df = pd.read_csv(REPO_ROOT / 'analysis' / 'aggregated.csv')
    df = df[df.batch_tag.isin(batch_tags)].copy()
    # Restrict to known panel members (drops claude / experimental short-names if any leaked in).
    df = df[df.model.isin(MODEL_PANEL) & df.target.isin(TARGETS)]
    return df

def cells_per_combo(df):
    """Per (model, target): count cells where the agent at least produced a scorer.json."""
    df = df.copy()
    df['has_scorer'] = df.engagement_status != 'no_scorer'
    return (df.groupby(['model','target']).has_scorer.sum()
              .unstack(fill_value=0)
              .reindex(index=MODEL_PANEL, columns=TARGETS, fill_value=0))

def shortfall(counts):
    return (TARGET_N - counts).clip(lower=0)

def write_matrix(rows, out_path):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=['model','target']).to_csv(out_path, index=False)
    return len(rows)

In [3]:
# Inspect current state — cells with scorer.json per (model, target).
df = load_panel(BATCH_TAGS)
print(f'aggregated rows: {len(df)} across {len(BATCH_TAGS)} batch(es)')
print()
counts = cells_per_combo(df)
print('cells with scorer.json per (model, target):')
print(counts)
print()
deficit = shortfall(counts)
needs_rerep = deficit[deficit.sum(axis=1) > 0]
print(f'combos below N={TARGET_N}:')
print(needs_rerep if not needs_rerep.empty else '  (none — proceed to phase 3)')

aggregated rows: 103 across 1 batch(es)

cells with scorer.json per (model, target):
target    josh  mesa  josh-mcp
model                         
sonnet       5     5         5
kimi         5     5         0
qwen         1     2         2
deepseek     5     5         5
glm          5     5         5
minimax      5     5         5
gemma        0     0         0
mistral      3     2         2
nemotron     5     5         5

combos below N=5:
target   josh  mesa  josh-mcp
model                        
kimi        0     0         5
qwen        4     3         3
gemma       5     5         5
mistral     2     3         3


In [4]:
# Build matrix-rerep.csv to bring every combo up to TARGET_N. Edit
# `out_path` if you'd like a different filename per iteration.
rerep_rows = []
for (m, t), reps in shortfall(counts).stack().items():
    if reps > 0:
        for _ in range(int(reps)):
            rerep_rows.append((m, t))

rerep_path = REPO_ROOT / 'orchestration' / 'matrix-rerep.csv'
n = write_matrix(rerep_rows, rerep_path)
print(f'wrote {n} cells → {rerep_path.relative_to(REPO_ROOT)}')
print()
from collections import Counter
for ((m, t), k) in sorted(Counter(rerep_rows).items()):
    print(f'  {m}/{t}: {k} reps')

wrote 38 cells → orchestration/matrix-rerep.csv

  gemma/josh: 5 reps
  gemma/josh-mcp: 5 reps
  gemma/mesa: 5 reps
  kimi/josh-mcp: 5 reps
  mistral/josh: 2 reps
  mistral/josh-mcp: 3 reps
  mistral/mesa: 3 reps
  qwen/josh: 4 reps
  qwen/josh-mcp: 3 reps
  qwen/mesa: 3 reps


### Apply the re-rep

```bash
BATCH_TAG=headline-rerep-$(date -u +%Y%m%d%H%M)
pixi run apply -- \
    --batch-tag "$BATCH_TAG" \
    --image-agent  ghcr.io/schmidtdse/josh-llm-experiment/fortree-agent:a5b3eae \
    --image-scorer ghcr.io/schmidtdse/josh-llm-experiment/fortree-scorer:a5b3eae \
    --matrix orchestration/matrix-rerep.csv

kubectl -n joshsim get jobs -l batch-tag=$BATCH_TAG -w

# Pull + re-aggregate across ALL batches so far:
pixi run pull $BATCH_TAG
pixi run aggregate runs/headline-stage1-20260601 runs/$BATCH_TAG
#                  ^^^ list every batch tag launched up to this point
```

Then **come back here, append the new tag to `BATCH_TAGS` in the config cell,
re-run the inspect cell** to check progress. Iterate this section until the
deficit table is empty.

## Smell-check — confirm `TARGET_N` per combo

When you think the re-rep loop is done, run this cell. It re-asserts the
deficit table is empty across every batch tag in `BATCH_TAGS`.

In [5]:
df = load_panel(BATCH_TAGS)
counts = cells_per_combo(df)
deficit = shortfall(counts)
remaining = int(deficit.sum().sum())
print(f'aggregated rows: {len(df)} across {len(BATCH_TAGS)} batch(es)')
print()
print('cells with scorer.json per (model, target):')
print(counts)
print()
if remaining == 0:
    print(f'✓ every combo at N >= {TARGET_N} ({len(MODEL_PANEL)*len(TARGETS)} combos). Ready for phase 3.')
else:
    print(f'✗ still {remaining} cells short of N={TARGET_N}. Re-rep again.')
    print(deficit[deficit.sum(axis=1) > 0])

aggregated rows: 103 across 1 batch(es)

cells with scorer.json per (model, target):
target    josh  mesa  josh-mcp
model                         
sonnet       5     5         5
kimi         5     5         0
qwen         1     2         2
deepseek     5     5         5
glm          5     5         5
minimax      5     5         5
gemma        0     0         0
mistral      3     2         2
nemotron     5     5         5

✗ still 38 cells short of N=5. Re-rep again.
target   josh  mesa  josh-mcp
model                        
kimi        0     0         5
qwen        4     3         3
gemma       5     5         5
mistral     2     3         3


## Phase 3 — stage 2 (advance combos with ≥1 pass)

Pass criterion: `substantive_conformance ∧ did_run ∧ regression_fit_ok` — the
full headline gate. A combo with even 1/5 passing in the re-rep'd `TARGET_N=5`
baseline advances with +10 more reps; 0/5 combos drop (we already have the
5-cell point estimate establishing 0%).

The re-rep batches and stage 1 are pooled for the pass count — re-rep cells
and stage-1 cells are methodologically identical, just submitted on different
days.

In [6]:
df = load_panel(BATCH_TAGS)
df['full_pass'] = (
    df.substantive_conformance.fillna(False).astype(bool)
    & df.did_run.fillna(False).astype(bool)
    & df.regression_fit_ok.fillna(False).astype(bool)
)
passes = (df.groupby(['model','target']).full_pass.sum()
            .unstack(fill_value=0)
            .reindex(index=MODEL_PANEL, columns=TARGETS, fill_value=0))
print('full passes per (model, target) — gates: substantive ∧ did_run ∧ regression_fit_ok')
print(passes)
print()

advancing = []
for m in MODEL_PANEL:
    for t in TARGETS:
        if int(passes.loc[m, t]) >= 1:
            advancing.append((m, t))

print(f'advancing combos ({len(advancing)} / {len(MODEL_PANEL)*len(TARGETS)}):')
for (m, t) in advancing:
    print(f'  ✓ {m}/{t}  ({int(passes.loc[m,t])}/{int(cells_per_combo(df).loc[m,t])} passed)')
print()
dropped = [(m, t) for m in MODEL_PANEL for t in TARGETS if (m, t) not in advancing]
print(f'dropped combos ({len(dropped)}) — 0/N passes, no advancement:')
for (m, t) in dropped:
    print(f'  ✗ {m}/{t}')

full passes per (model, target) — gates: substantive ∧ did_run ∧ regression_fit_ok
target    josh  mesa  josh-mcp
model                         
sonnet       5     5         5
kimi         5     4         0
qwen         1     2         2
deepseek     3     5         4
glm          5     5         5
minimax      1     3         2
gemma        0     0         0
mistral      3     2         0
nemotron     0     0         0

advancing combos (19 / 27):
  ✓ sonnet/josh  (5/5 passed)
  ✓ sonnet/mesa  (5/5 passed)
  ✓ sonnet/josh-mcp  (5/5 passed)
  ✓ kimi/josh  (5/5 passed)
  ✓ kimi/mesa  (4/5 passed)
  ✓ qwen/josh  (1/1 passed)
  ✓ qwen/mesa  (2/2 passed)
  ✓ qwen/josh-mcp  (2/2 passed)
  ✓ deepseek/josh  (3/5 passed)
  ✓ deepseek/mesa  (5/5 passed)
  ✓ deepseek/josh-mcp  (4/5 passed)
  ✓ glm/josh  (5/5 passed)


  ✓ glm/mesa  (5/5 passed)
  ✓ glm/josh-mcp  (5/5 passed)
  ✓ minimax/josh  (1/5 passed)
  ✓ minimax/mesa  (3/5 passed)
  ✓ minimax/josh-mcp  (2/5 passed)
  ✓ mistral/josh  (3/3 passed)
  ✓ mistral/mesa  (2/2 passed)

dropped combos (8) — 0/N passes, no advancement:
  ✗ kimi/josh-mcp
  ✗ gemma/josh
  ✗ gemma/mesa
  ✗ gemma/josh-mcp
  ✗ mistral/josh-mcp
  ✗ nemotron/josh
  ✗ nemotron/mesa
  ✗ nemotron/josh-mcp


In [7]:
STAGE2_REPS = 10
stage2_rows = [(m, t) for (m, t) in advancing for _ in range(STAGE2_REPS)]
stage2_path = REPO_ROOT / 'orchestration' / 'matrix-stage2.csv'
n = write_matrix(stage2_rows, stage2_path)
print(f'wrote {n} cells ({len(advancing)} combos × {STAGE2_REPS} reps) → {stage2_path.relative_to(REPO_ROOT)}')

wrote 190 cells (19 combos × 10 reps) → orchestration/matrix-stage2.csv


### Apply stage 2

```bash
BATCH_TAG=headline-stage2-$(date -u +%Y%m%d)
pixi run apply -- \
    --batch-tag "$BATCH_TAG" \
    --image-agent  ghcr.io/schmidtdse/josh-llm-experiment/fortree-agent:a5b3eae \
    --image-scorer ghcr.io/schmidtdse/josh-llm-experiment/fortree-scorer:a5b3eae \
    --matrix orchestration/matrix-stage2.csv

kubectl -n joshsim get jobs -l batch-tag=$BATCH_TAG -w

# Pull + final aggregation across every batch tag the headline used:
pixi run pull $BATCH_TAG
pixi run aggregate runs/headline-stage1-20260601 runs/headline-rerep-* runs/$BATCH_TAG
```

Some stage-2 cells will also fail at `FailJob`. The (model, target) was
already established as a passing combo in stage 1, so the methodologically
consistent move is to **re-rep stage 2 failures back to its target N**
(typically 10–15 cumulative cells per advancing combo) using the same loop
in phase 2 — just set `TARGET_N = 15` and re-iterate the re-rep cells against
the full batch-tag list.

## Done — render the headline notebook

```bash
pixi run lab
# In the lab UI: open analysis/headline.ipynb, set BATCH_FILTER to the full
# list of headline batch tags, Run All. Panels A (substantive pass rate +
# Wilson CI) and B (5-axis heatmap) will render across the combined panel.
```

The headline notebook treats every batch tag as one panel — the 5-axis
heatmap renders one dot per replicate per tile, so a combo that ran in
stage 1 (5 cells) + stage 2 (10 cells) shows 15 dots; a dropped combo shows
its 5 stage-1 dots only.